In [ ]:

# =================== 2. 【关键】转换为 JAX 稀疏矩阵 ===================
print("正在转换为 JAX BCOO 格式 (以便 GPU 加速)...")
# BCOO 是 JAX 在 GPU 上最高效的稀疏格式
# 这一步将矩阵数据搬运到 GPU 显存中，且只占用几 MB
H_jax = jsparse.BCOO.from_scipy_sparse(H_gray_sparse)
# 排序索引以获得最佳稀疏乘法性能
H_jax = H_jax.sort_indices()
print("JAX 稀疏矩阵准备就绪。")



In [ ]:
# =================== 5. 训练循环 ===================
n_params = depth * n_qubits
init_params = jnp.zeros(n_params)

# 使用 JAX 的优化器
optimizer = optax.adam(learning_rate=0.01)
opt_state = optimizer.init(init_params)
params = init_params

print(f"\n🚀 开始在 A100 上训练 | Depth: {depth} | Qubits: {n_qubits}")
print("预编译中 (Warm-up)...")
# 触发一次编译
_ = jax.value_and_grad(loss_fn)(params)
print("编译完成，开始跑！")

start_total = time.time()

for i in range(steps):
    step_start = time.time()

    # 计算梯度和能量
    value, grads = jax.value_and_grad(loss_fn)(params)

    # 更新参数
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

    if i % 50 == 0:
        # block_until_ready() 用于等待 GPU 异步计算完成，获取准确时间
        grads.block_until_ready()
        t_sec = time.time() - step_start
        print(f"Step {i:4d} | Energy: {value:.8f} Ha | Time: {t_sec*1000:.2f} ms")

print(f"训练完成，总耗时: {time.time()-start_total:.2f}s")